# FRESEAN Analysis of HEWL using IMDv3 streaming (coarse-grained, solution, streaming, ~300 K)

Below, we provide a **coarse-grained** FRESEAN example using pyFRESEAN (`import pyfresean`). The idea is to load a raw solvated protein trajectory using IMDv3-streaming, "select" just the protein atoms, unwrap it on the fly with MDAnalysis, coarse-grain it into memory, run FRESEAN on the CG representation, visualize the total vibrational density of states (VDoS), and inspect per-mode spectra contributions for **CG modes 7 and 8**.

Unlike [`02_AA_CG-hewl-solution-300K.ipynb`](02_AA_CG-hewl-solution-300K.ipynb), we skip GROMACS postprocess (`postprocess-prot.sh`), which is done using MDAnalysis's `Unwrap` transformation. This steup bypasses the need to save expensive all-atom trajectory files on disk. At the same time, it must be noted that a data stream can onyl be accessed once, and thus any relevant files thatr might be needed in the future must be saved to disk.

We provide some theoretical explanations where relevant, however, one can refer to the original [FRESEAN paper](https://pubs.acs.org/doi/10.1021/acs.jctc.2c01309) and [Streaming paper](https://pubs.acs.org/doi/10.1021/acs.jctc.6c01055) for more details.

This notebook needs to be run from the `examples/` directory. It expects HEWL input files under `input_data/MD-HEWL-303K/` (see section 1).

## Setup — paths and analysis parameters

Here, we define some basic parameters for the analysis.
These parameters are shared by the coarse-graining (CG) step and the FRESEAN step.

- **`START` / `STOP` / `STEP`** — trajectory slice (MDAnalysis-style: `START` inclusive, `STOP` exclusive). Set `STOP = None` to use all frames. This window is used when building the CG universe; FRESEAN then runs over the full in-memory CG trajectory.
- **`N_CORR`** — number of frequency bins in the output spectrum (and maximum lag in the correlation FFT).
- **`DT`** — time between saved frames (ps); sets the frequency grid spacing together with `N_CORR`.
- **`SIGMA`** — Gaussian broadening width (cm $^{-1}$) applied in frequency space; used for smoothing the spectrum.
- **`MODE_NUMBERS = (7, 8)`** — CG modes of interest; commonly used in enhanced sampling protocols.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np

from pyfresean import Align, CoarseGrain, FRESEAN, Unwrap
from pyfresean.postprocess import mode_spectra, plot_spectra

INPUT_DATA = Path("input_data") / "MD-HEWL-303K"
OUTPUT_DATA = Path("output_data") / "MD-HEWL-303K-streaming"
OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

# trajectory window (passed to CG construction)
START = 0        # first frame index (inclusive)
STOP = 1000      # last frame index (exclusive); None = use all frames
STEP = 1         # frame stride between analyzed frames

# FRESEAN spectral parameters
N_CORR = 100     # number of frequency bins (= max lag in the correlation FFT)
DT = 0.02        # ps between consecutive trajectory frames
SIGMA = 10.0     # cm^-1 Gaussian broadening applied in frequency space

# CG modes to inspect (1-based numbering)
MODE_NUMBERS = (7, 8)
MODE_INDICES = tuple(m - 1 for m in MODE_NUMBERS)
MAX_FREQ_PLOT = 200.0  # cm^-1, x-axis limit for VDoS plots

## 1. Loading the all-atom trajectory

We read the **raw solvated** trajectory generated from unbiased NPT MD in water:

- `run.tpr` — GROMACS topology from `grompp`
- `run.trr` — trajectory from `mdrun`

Instead of writing a protein-only, PBC-corrected TRR with `gmx trjconv -pbc mol`, we register **`Unwrap(protein)`** as a trajectory transformation so molecules are made whole as frames are read. These files must be generated with GROMACS ([`run-slurm.sh`](input_data/MD-HEWL-303K/run-slurm.sh)) if they are not already present (see [`input_data/README.md`](input_data/README.md)).

In [ ]:
# defining the source paths to the all-atom topology and trajectory
topol = INPUT_DATA / "output" / "run.tpr"
traj = INPUT_DATA / "output" / "run.trr"

for path in (topol, traj):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Generate HEWL solution MD first "
            f"(see input_data/MD-HEWL-303K/run-slurm.sh). "
            f"This notebook does not need postprocess-prot.sh."
        )

# loading the trajectory and selecting the protein atoms
u_aa = mda.Universe(str(topol), str(traj))
protein = u_aa.select_atoms("protein")

# unwrap protein fragments as frames are read
u_aa.trajectory.add_transformations(Unwrap(protein))

# number of frames to use
n_use = (len(u_aa.trajectory) if STOP is None else STOP) - START
print(f"System atoms: {u_aa.atoms.n_atoms}")
print(f"Protein atoms: {protein.n_atoms}")
print(f"Residues: {protein.n_residues}")
print(f"Trajectory frames: {len(u_aa.trajectory)} (using [{START}:{STOP}:{STEP}], ~{n_use // STEP} frames)")

## 2. Coarse-graining the trajectory

`CoarseGrain.cg_universe` maps each residue to one or two center-of-mass (COM) beads:

- **BACK** — backbone COM
- **SIDE** — sidechain COM (skipped for GLY and terminal caps)

Bead positions and velocities are mass-weighted averages of the constituent atoms. The CG trajectory has fewer degrees of freedom ($3 N_{\mathrm{beads}}$ vs $3 N_{\mathrm{atoms}}$), so high-frequency internal vibrations are not represented explicitly.

Because `Unwrap` is already registered on `u_aa`, each frame is made whole before bead COMs and velocities are computed. With `in_memory=True`, the CG universe is kept in memory so FRESEAN can iterate it freely after the all-atom stream has been read.

`centered_mode="ref"` expresses internal coordinates relative to a **reference** frame (no per-frame tracking). The mapping's `n_constraints` (0 here for solution) is passed through to the CG FRESEAN run.

In [ ]:
cg, u_cg = CoarseGrain.cg_universe(
    u_aa,
    select="protein",
    start=START,
    stop=STOP,
    step=STEP,
    output_cg_topology=None,  # set to None to avoid writing to disk and improve speed
    output_cg_trajectory=None,  # set to None to avoid writing to disk and improve speed
    centered_mode="ref",  # reference-frame internal coords (no per-frame tracking)
    in_memory=True,
)

print(f"CG beads: {cg.mapping.n_beads}")
print(f"n_constraints (CG): {cg.mapping.n_constraints}")
print(f"CG frames: {len(u_cg.trajectory)}")

## 3. Run FRESEAN on the coarse-grained trajectory

Before `FRESEAN`, we align each CG frame to the **first-frame** bead positions, using it as a reference. This removes rigid-body motion from the trajectory used in the correlation analysis, as in [Notebook 01](01_AA-alanine-dipeptide-gas-300K.ipynb).

Calling `FRESEAN.run()` follows the same workflow as [Notebook 01](01_AA-alanine-dipeptide-gas-300K.ipynb): collect mass-weighted velocities, build the spectral correlation matrix $\mathbf{C}(\omega)$, and diagonalize at each frequency. We use 0 here for the number of constraints so the VDoS normalization matches the solution-phase CG model.

Results are stored in `analysis_cg.results`: `freqs`, `vdos_total`, `eigenvectors`, `corr_matrix`, etc.

In [ ]:
# aligning CG frames to the first-frame bead positions
u_cg.trajectory[0]
ref_cg = u_cg.atoms.positions.copy()
u_cg.trajectory.add_transformations(
    Align(
        u_cg.atoms,
        reference_positions=ref_cg,
        place_com_in_box=False,
    ),
)

analysis_cg = FRESEAN(
    u_cg,
    select="all",
    n_constraints=int(cg.mapping.n_constraints),
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
    # compute_modes=[0.0], # this option lets you pick a set of frequencies to run instead of entire spectrum
)
analysis_cg.run()

freqs_cg = analysis_cg.results.freqs
vdos_cg = analysis_cg.results.vdos_total
eigenvectors_cg = analysis_cg.results.eigenvectors
corr_cg = analysis_cg.results.corr_matrix

## 4. Plot total VDoS — coarse-grained

Total vibrational density of states $g_{\mathrm{CG}}(\omega)$ from the coarse-grained trajectory.

In [ ]:
print(f"T (CG) = {analysis_cg.results.avg_temperature:.1f} K")

fig, ax = plt.subplots(figsize=(7, 4))
plot_spectra(
    freqs_cg,
    vdos_cg,
    ax=ax,
    xlim=(0, MAX_FREQ_PLOT),
    labels="CG VDoS",
    colors="tab:orange",
    title="Total VDoS (streaming): coarse-grained",
)
plt.tight_layout()
plt.show()

## 5. Per-mode VDoS — CG modes 7 and 8

Project CG modes 7 and 8 at zero frequency onto $C_{\mathrm{CG}}(\omega)$ with `mode_spectra`, as in notebook 02.

In [ ]:
freq_idx = 0  # zero frequency
cg_modes = eigenvectors_cg[freq_idx, MODE_INDICES]
mode_vdos = mode_spectra(corr_cg, cg_modes)

fig, ax = plt.subplots(figsize=(6, 4))
colors = ["tab:blue", "tab:green"]
for i, mode_num in enumerate(MODE_NUMBERS):
    plot_spectra(
        freqs_cg,
        mode_vdos[i],
        ax=ax,
        labels=f"CG mode {mode_num}",
        colors=colors[i],
    )
ax.set_xlim(0, MAX_FREQ_PLOT)
ax.set_title(f"CG modes {MODE_NUMBERS} at {freqs_cg[freq_idx]:.1f} cm$^{{-1}}$")
plt.tight_layout()
plt.show()